# DG-HMCF Demo Notebook

This notebook demonstrates:
1. Loading the DG-HMCF model
2. Running inference on a synthetic batch
3. Interpreting explainability outputs
4. Visualising modality reliability weights and attention maps


In [ ]:
import sys, os
# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import yaml

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Load Configuration

In [ ]:
with open('../configs/base_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

model_config = config['model']
print('Loaded config. Fusion dim:', model_config['fusion_dim'])

## 2. Initialise Model

We skip loading pretrained Wav2Vec2/RoBERTa/ViT weights here for speed 
(set `freeze_*=True` and use random initialisations for the demo).

In [ ]:
from models.dg_hmcf import DGHMCF

device = torch.device('cpu')
model = DGHMCF(model_config).to(device)
model.eval()

param_counts = model.count_parameters()
print(f'Total trainable parameters: {param_counts["total"]:,}')
for name, count in param_counts.items():
    if name != 'total':
        print(f'  {name:<25}: {count:>10,}')

## 3. Create a Synthetic Batch

Simulates a batch of 4 samples with varying modality availability.

In [ ]:
B = 4  # batch size
embed_dim = model_config['speech_embed_dim']
max_audio = model_config['speech']['max_length']
max_text = model_config['text']['max_length']
max_frames = 32   # reduced for demo speed
max_eeg_segs = 10
n_eeg_ch = model_config['eeg']['n_channels']
seg_len = model_config['eeg']['segment_length']

# Modality mask: each sample has a different subset
# [speech, text, face, eeg]
modality_masks = torch.tensor([
    [1, 1, 1, 0],   # sample 0: speech+text+face
    [1, 1, 0, 1],   # sample 1: speech+text+eeg
    [1, 0, 0, 1],   # sample 2: speech+eeg only
    [0, 1, 1, 1],   # sample 3: text+face+eeg
], dtype=torch.float32)

batch = {
    'modality_mask': modality_masks,
    'label': torch.tensor([1, 0, 1, 0], dtype=torch.long),
    'phq8_score': torch.tensor([0.60, 0.20, 0.58, 0.15], dtype=torch.float32),
    'subject_id': ['P001', 'P002', 'P003', 'P004'],
    'speech': {
        'waveform': torch.randn(B, max_audio),
        'attention_mask': torch.ones(B, max_audio),
        'behavioral_features': torch.randn(B, 6),
    },
    'text': {
        'input_ids': torch.randint(0, 50265, (B, max_text)),
        'attention_mask': torch.ones(B, max_text),
        'linguistic_features': torch.randn(B, 5),
    },
    'face': {
        'pixel_values': torch.randn(B, max_frames, 3, 224, 224),
        'frame_mask': torch.ones(B, max_frames),
        'behavioral_features': torch.randn(B, 7),
    },
    'eeg': {
        'segments': torch.randn(B, max_eeg_segs, n_eeg_ch, seg_len),
        'segment_mask': torch.ones(B, max_eeg_segs),
    },
}

print('Batch created successfully.')
print('Modality masks:')
for i, mask in enumerate(modality_masks):
    present = [['speech','text','face','eeg'][j] for j, v in enumerate(mask) if v > 0]
    print(f'  Sample {i}: {", ".join(present)}')

## 4. Forward Pass

In [ ]:
with torch.no_grad():
    outputs = model(batch)

print('Output keys:', list(outputs.keys()))
print('Classification logits shape:', outputs['classification_logits'].shape)
print('PHQ-8 scores (normalised):', outputs['phq8_score'].numpy().round(3))
print('PHQ-8 scores (raw):', (outputs['phq8_score'] * 24).numpy().round(1))
print('Reliability weights shape:', outputs['reliability_weights'].shape)

## 5. Predictions

In [ ]:
probs = torch.softmax(outputs['classification_logits'], dim=-1)
preds = probs.argmax(dim=-1)

print('\nPrediction Summary:')
print(f'{"Subject":<10} {"True":<8} {"Pred":<8} {"P(dep)":<10} {"PHQ-8":<8} {"Correct"}')
print('-' * 55)
for i in range(B):
    print(
        f'{batch["subject_id"][i]:<10} '
        f'{int(batch["label"][i]):<8} '
        f'{int(preds[i]):<8} '
        f'{probs[i,1].item():.3f}      '
        f'{outputs["phq8_score"][i].item()*24:.1f}     '
        f'{"✓" if preds[i] == batch["label"][i] else "✗"}'
    )

## 6. Reliability Weights Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

weights = outputs['reliability_weights'].numpy()
modality_names = ['Speech', 'Text', 'Face', 'EEG']

fig, axes = plt.subplots(1, B, figsize=(16, 4), sharey=True)
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

for i, ax in enumerate(axes):
    w = weights[i]
    bars = ax.bar(modality_names, w, color=colors, alpha=0.85, edgecolor='black')
    ax.set_title(f'Sample {i} ({batch["subject_id"][i]})\n'
                 f'True: {int(batch["label"][i])}, Pred: {int(preds[i])}',
                 fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Reliability Weight', fontsize=9)
    for bar, val in zip(bars, w):
        if val > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Dynamic Reliability Weights per Sample', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Modality Importance Visualisation

In [ ]:
importance = outputs['modality_importance'].numpy()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(modality_names, importance.mean(axis=0), color=colors,
              alpha=0.85, edgecolor='black')
ax.errorbar(modality_names, importance.mean(axis=0),
            yerr=importance.std(axis=0),
            fmt='none', color='black', capsize=5)
ax.set_xlabel('Modality', fontsize=12)
ax.set_ylabel('Mean Importance Score', fontsize=12)
ax.set_title('Aggregate Modality Importance', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Explainability Report

In [ ]:
# Generate human-readable report for first sample
report = model.explainability.generate_report(
    explainability_output={
        'modality_importance': outputs['modality_importance'],
        'reliability_weights': outputs['reliability_weights'],
        'embedding_norms': outputs['reliability_weights'],  # proxy for demo
        'attention_maps': outputs['attention_maps'],
    },
    prediction={
        'classification_logits': outputs['classification_logits'],
        'phq8_score': outputs['phq8_score'],
    },
    sample_idx=0,
)

print('=' * 60)
print('EXPLAINABILITY REPORT — Sample P001')
print('=' * 60)
print(f'Prediction   : {report["prediction"]}')
print(f'Confidence   : {report["confidence"]:.1%}')
print(f'Est. PHQ-8   : {report["phq8_score_estimated"]}')
print(f'Severity     : {report["severity"]}')
print()
print('Modality Importance Ranking:')
for item in report['modality_importance_ranking']:
    print(f'  {item["modality"]:<10}: {item["importance"]:.4f}')
print()
print('Summary:')
print(report['summary'])

## 9. Cross-Modal Attention Maps

In [ ]:
attn_maps = outputs['attention_maps']

if attn_maps:
    n = len(attn_maps)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
    if n == 1:
        axes = [axes]
    for ax, (pair_name, attn) in zip(axes, attn_maps.items()):
        val = attn[0].cpu().numpy().squeeze()  # first sample
        if val.ndim == 0:
            val = np.array([[float(val)]])
        elif val.ndim == 1:
            val = val.reshape(1, -1)
        im = ax.imshow(val, cmap='Blues', aspect='auto')
        ax.set_title(pair_name.replace('_', ' '), fontsize=9)
        plt.colorbar(im, ax=ax)
    plt.suptitle('Cross-Modal Attention Weights', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No attention maps available (all pairs were absent).')

## 10. Quick Preprocessing Demo

In [ ]:
# Demonstrate text and EEG preprocessors with synthetic data
from data.preprocessing.text_preprocessor import TextPreprocessor
from data.preprocessing.eeg_preprocessor import EEGPreprocessor

# --- Text ---
text_preprocessor = TextPreprocessor(max_length=64)
sample_transcript = """
Ellie: How have you been feeling lately?
P: I don't know. Kind of empty, I guess. I haven't felt like doing anything.
   Everything feels pointless. I'm really tired all the time.
Ellie: How long have you been feeling this way?
P: Maybe two or three weeks. I can't really remember.
"""
text_output = text_preprocessor.preprocess(sample_transcript, remove_interviewer=True)
ling_feats = text_output['linguistic_features']
feature_names = ['Sentiment', 'Neg-word ratio', 'Uncertainty', 'Self-reference', 'Emotional polarity']
print('Linguistic Features:')
for name, val in zip(feature_names, ling_feats):
    print(f'  {name:<22}: {val:.4f}')

print()

# --- EEG ---
eeg_preprocessor = EEGPreprocessor(sampling_rate=256, n_channels=32, segment_length=256,
                                     max_segments=10, overlap=0.5)
fake_eeg = np.random.randn(32, 2560).astype(np.float32)  # 10 seconds
eeg_output = eeg_preprocessor.preprocess(fake_eeg)
print(f'EEG segments shape: {eeg_output["segments"].shape}')
print(f'Real segments: {int(eeg_output["segment_mask"].sum())}')

---

## Summary

This notebook demonstrated:
- Model instantiation and parameter counting
- Batch construction with variable modality availability
- Forward pass producing classification + PHQ-8 outputs
- Dynamic reliability weight visualisation
- Modality importance analysis
- Human-readable explainability reports
- Cross-modal attention map visualisation
- Linguistic and EEG preprocessing

For full training, see `scripts/train.py`.